# 問題
複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数collateを実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

[{'text': 'hide new secretions from the parental units',<br>
  'label': tensor([0.]),<br>
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},<br>
 {'text': 'contains no wit , only labored gags',<br>
  'label': tensor([0.]),<br>
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},<br>
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',<br>
  'label': tensor([1.]),<br>
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},<br>
 {'text': 'remains utterly satisfied to remain the same throughout',<br>
  'label': tensor([0.]),<br>
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]<br>

collate関数を通した結果は以下のようになることが想定される。

{'input_ids': tensor([<br>
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],<br>
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],<br>
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],<br>
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),<br>
 'label': tensor([<br>
    [1.],<br>
    [0.],<br>
    [0.],<br>
    [0.]])}<br>


In [1]:
# 単語埋め込み語彙の作成
import numpy as np
from gensim.models import KeyedVectors
import torch

model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
vocab = list(model.key_to_index.keys())
d_emb = model.vector_size
V = len(vocab) + 1

# 埋め込み行列の初期化
E = np.zeros((V, d_emb), dtype=np.float32)

# インデックス対応表
word2id = {'<PAD>': 0}
id2word = {0: '<PAD>'}

# 行列にベクトルを格納
for i, word in enumerate(vocab, start=1):
    E[i] = model[word]
    word2id[word] = i
    id2word[i] = word

In [ ]:
import torch

def sst_build_answer_batch(path: str):
    """
    SST-2のTSVを読み込み、
      - 文章→単語分割
      - word2idでID列に変換（辞書にない語は除外）
      - 最長系列長に合わせて0埋めパディング
      - トークン列の長い順にソート
      - 最終的に input_ids と label をそれぞれTensorとしてまとめて返す
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            row = raw.strip().split("\t")
            if len(row) < 2:
                continue
            label = row[1]
            if label not in ("0", "1"):
                continue
            text = row[0]
            tokens = text.strip().split()
            examples.append({"text": text, "label": int(label), "tokens": tokens})

    # ID化
    kept_examples = []
    for ex in examples:
        ids = [word2id[w] for w in ex["tokens"] if w in word2id]
        if ids:
            ex["input_ids"] = ids
            kept_examples.append(ex)
    print(kept_examples)

    # --- 長い順にソート ---
    kept_examples = sorted(kept_examples, key=lambda x: len(x["input_ids"]), reverse=True)

    # --- パディング ---
    max_len = len(kept_examples[0]["input_ids"])
    padded_ids = []
    labels = []

    for ex in kept_examples:
        ids = ex["input_ids"]
        padded = ids + [0] * (max_len - len(ids))
        padded_ids.append(padded)
        labels.append([float(ex["label"])])  # [[1.], [0.], ...]

    # --- Tensor化 ---
    input_tensor = torch.tensor(padded_ids, dtype=torch.long)
    label_tensor = torch.tensor(labels, dtype=torch.float)

    # --- dict形式で返す ---
    batch = {
        "input_ids": input_tensor,
        "label": label_tensor
    }

    return batch
# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"
dev_71 = sst_build_answer_batch(path_dev)
train_71 = sst_build_answer_batch(path_train)

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from typing import List, Dict, Any

def collate(examples: List[Dict[str, Any]]):
    # 1) 長い順にソート
    examples = sorted(examples, key=lambda x: len(x["input_ids"]), reverse=True)

    seqs = []
    labels = []

    for ex in examples:
        ids = ex["input_ids"]
        # 2) ids を必ず LongTensor に
        if isinstance(ids, torch.Tensor):
            ids_t = ids.long()
        else:  # list 等
            ids_t = torch.tensor(ids, dtype=torch.long)
        seqs.append(ids_t)

        # 3) label を (1,) float tensor に統一
        lab = ex.get("label")
        if isinstance(lab, torch.Tensor):
            lab_t = lab.float().view(-1)        # 例: tensor([0.]) -> (1,)
        else:
            lab_t = torch.tensor([float(lab)], dtype=torch.float)
        labels.append(lab_t)

    # 4) 0 でパディング（batch_first=True → (N, L)）
    input_ids = pad_sequence(seqs, batch_first=True, padding_value=0)   # (N, L)
    label = torch.stack(labels, dim=0)                                  # (N, 1)

    return {"input_ids": input_ids, "label": label}

In [ ]:
import torch

test = [
    {'text': 'hide new secretions from the parental units',
     'label': torch.tensor([0.]),
     'input_ids': torch.tensor([5785, 66, 113845, 18, 12, 15095, 1594])},

    {'text': 'contains no wit , only labored gags',
     'label': torch.tensor([0.]),
     'input_ids': torch.tensor([3475, 87, 15888, 90, 27695, 42637])},

    {'text': 'that loves its characters and communicates something rather beautiful about human nature',
     'label': torch.tensor([1.]),
     'input_ids': torch.tensor([4, 5053, 45, 3305, 31647, 348, 904, 2815, 47, 1276, 1964])},

    {'text': 'remains utterly satisfied to remain the same throughout',
     'label': torch.tensor([0.]),
     'input_ids': torch.tensor([987, 14528, 4941, 873, 12, 208, 898])}
]

result = collate(test)
print(result)

{'input_ids': tensor([[     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,
           1276,   1964],
        [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,
              0,      0],
        [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,
              0,      0],
        [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,
              0,      0]]), 'label': tensor([[1.],
        [0.],
        [0.],
        [0.]])}
